<a href="https://colab.research.google.com/github/AnjanPayra/MEM-FET-Essential-protein-prediction-using-membership-feature-and-machine-learning-approach/blob/main/MEM_FET_Updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.feature_selection import RFE
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, mean_squared_error, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report
)

warnings.filterwarnings("ignore")
np.random.seed(42)

UPLOAD_DIR = "/content"
OUT_DIR = "/mnt/user-data/outputs"
os.makedirs(OUT_DIR, exist_ok=True)

DATASETS = ["YDIP", "YHQ", "YMBD", "YMIPS"]

# ----------------------------------------------------------------------
# 0. Load essential-protein reference list (ground truth, positive class)
# ----------------------------------------------------------------------
def load_essential_set(upload_dir):
    e = pd.read_excel(os.path.join(upload_dir, "Essential.xlsx"), header=None)
    ess = set(e[0].astype(str).str.strip())
    return ess

In [12]:
# ----------------------------------------------------------------------
# 1. Load subcellular-localization annotation -> set of compartments/protein
# ----------------------------------------------------------------------
COMPARTMENT_KEYWORDS = [
    "nucleus", "nucleolus", "cytoplasm", "cytosol", "mitochondri",
    "golgi", "endoplasmic reticulum", "vacuole", "bud", "cell membrane",
    "membrane", "peroxisome", "cell wall", "ribosome", "endosome",
    "chromosome", "spindle", "nuclear", "secreted", "lipid particle"
]

def load_subcellular_map(upload_dir):
    sc = pd.read_csv(os.path.join(upload_dir, "Sub_cellular.csv"))
    sc.columns = ["gene", "location_text"]
    sc = sc.dropna(subset=["gene"])
    sl_map = {}
    for _, row in sc.iterrows():
        gene = str(row["gene"]).strip()
        text = str(row["location_text"]).lower()
        found = {kw for kw in COMPARTMENT_KEYWORDS if kw in text}
        if found:
            sl_map[gene] = found
    return sl_map

In [13]:

# ----------------------------------------------------------------------
# 2. Load precomputed GO_Nb(u) score  (…_GO_final.csv : protein,score)
# ----------------------------------------------------------------------
def load_go_scores(dataset, upload_dir):
    path = os.path.join(upload_dir, f"{dataset}_GO_final.csv")
    go = pd.read_csv(path, header=None, names=["protein", "GO_Nb"])
    go["protein"] = go["protein"].astype(str).str.strip()
    return dict(zip(go["protein"], go["GO_Nb"]))

In [14]:

# ----------------------------------------------------------------------
# 3. Build PPI graph
# ----------------------------------------------------------------------
def load_network(dataset, upload_dir):
    path = os.path.join(upload_dir, f"{dataset}.txt")
    df = pd.read_csv(path)
    df.columns = ["p1", "p2"]
    df = df.dropna()
    df = df[df["p1"].astype(str).str.strip() != df["p2"].astype(str).str.strip()]
    G = nx.Graph()
    G.add_edges_from(zip(df["p1"].astype(str).str.strip(), df["p2"].astype(str).str.strip()))
    G.remove_edges_from(nx.selfloop_edges(G))
    return G

In [5]:
# ----------------------------------------------------------------------
# 4. Feature computation :  CC, ECC, SL_Nb
# ----------------------------------------------------------------------
def compute_cc(G):
    """Clustering coefficient CC_i, per Fig.9 metric table."""
    return nx.clustering(G)


def compute_ecc(G):
    """
    Edge Clustering Coefficient:
        ECC(u,v) = triangles(u,v)^3-ish term / max(deg(u), deg(v))  [Fig.9]
        ECC(u)   = sum over level-1 neighbours t of ECC(u,t)
    We use the standard edge-clustering-coefficient definition:
        ECC(u,v) = (# triangles containing edge u-v) / max(deg(u)-1, deg(v)-1)
    """
    ecc_edge = {}
    deg = dict(G.degree())
    for u, v in G.edges():
        common = len(list(nx.common_neighbors(G, u, v)))
        denom = max(deg[u] - 1, deg[v] - 1)
        ecc_edge[(u, v)] = common / denom if denom > 0 else 0.0

    ecc_node = {n: 0.0 for n in G.nodes()}
    for (u, v), val in ecc_edge.items():
        ecc_node[u] += val
        ecc_node[v] += val
    return ecc_node


def compute_sl_nb(G, sl_map):
    """
    Localized Significance Score (subcellular-localization based), mirrors
    the GO_Nb construction in Fig.9 but built from subcellular compartments:
        SL_Nb(u,v) = |Union(SL(t)) for t in common_neigh(u,v)|^2 / (|SL(u)|*|SL(v)|)
        SL_Nb(u)   = sum over level-1 neighbours t of SL_Nb(u,t)
    """
    sl_edge = {}
    for u, v in G.edges():
        su, sv = sl_map.get(u), sl_map.get(v)
        if su and sv:
            common = list(nx.common_neighbors(G, u, v))
            union_locs = set()
            for t in common:
                union_locs |= sl_map.get(t, set())
            r = len(union_locs) ** 2
            s = len(su) * len(sv)
            sl_edge[(u, v)] = r / s if s > 0 else 0.0
        else:
            sl_edge[(u, v)] = 0.0

    sl_node = {n: 0.0 for n in G.nodes()}
    for (u, v), val in sl_edge.items():
        sl_node[u] += val
        sl_node[v] += val
    return sl_node


# ----------------------------------------------------------------------
# 5. MEM-FET score  (Fig.9 formula)
#    MEM-FET(phi) = ECC(phi) + MAX(CC,GO_Nb)/MAX(ECC,CC,GO_Nb)
# ----------------------------------------------------------------------
def compute_mem_fet(row):
    ecc, cc, go_nb = row["ECC"], row["CC"], row["GO_Nb"]
    denom = max(ecc, cc, go_nb)
    if denom == 0:
        return ecc
    return ecc + (max(cc, go_nb) / denom)




In [15]:
# ----------------------------------------------------------------------
# 6. Build the full per-protein feature table for one dataset
# ----------------------------------------------------------------------
def build_feature_table(dataset, essential_set, sl_map, upload_dir):
    print(f"\n[{dataset}] building network & features ...")
    G = load_network(dataset, upload_dir)
    print(f"  nodes={G.number_of_nodes()}  edges={G.number_of_edges()}")

    cc = compute_cc(G)
    ecc = compute_ecc(G)
    sl_nb = compute_sl_nb(G, sl_map)
    go_scores = load_go_scores(dataset, upload_dir)

    rows = []
    for n in G.nodes():
        rows.append({
            "protein": n,
            "CC": cc.get(n, 0.0),
            "ECC": ecc.get(n, 0.0),
            "SL_Nb": sl_nb.get(n, 0.0),
            "GO_Nb": go_scores.get(n, 0.0),
            "degree": G.degree(n),
            "essential": 1 if n in essential_set else 0,
        })
    df = pd.DataFrame(rows)
    df["MEM_FET"] = df.apply(compute_mem_fet, axis=1)
    return df

In [7]:
# ----------------------------------------------------------------------
# 7. Feature ranking (step 2 of flow diagram): RF, XGBoost, LR + RFE
# ----------------------------------------------------------------------
def rank_features(df, feature_cols):
    X = df[feature_cols].values
    y = df["essential"].values

    scaler = MinMaxScaler()
    Xs = scaler.fit_transform(X)

    rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42)
    rf.fit(Xs, y)
    rf_importance = dict(zip(feature_cols, rf.feature_importances_))

    xgb = XGBClassifier(
        n_estimators=300, max_depth=4, eval_metric="logloss",
        scale_pos_weight=(y == 0).sum() / max((y == 1).sum(), 1),
        random_state=42, verbosity=0
    )
    xgb.fit(Xs, y)
    xgb_importance = dict(zip(feature_cols, xgb.feature_importances_))

    lr = LogisticRegression(max_iter=2000, class_weight="balanced")
    rfe = RFE(lr, n_features_to_select=1)
    rfe.fit(Xs, y)
    rfe_rank = dict(zip(feature_cols, rfe.ranking_))  # 1 = best

    rank_table = pd.DataFrame({
        "feature": feature_cols,
        "RF_importance": [rf_importance[f] for f in feature_cols],
        "XGB_importance": [xgb_importance[f] for f in feature_cols],
        "RFE_rank(1=best)": [rfe_rank[f] for f in feature_cols],
    })
    # combined score: higher RF/XGB importance and lower RFE rank is better
    rank_table["combined_score"] = (
        rank_table["RF_importance"].rank(ascending=False) +
        rank_table["XGB_importance"].rank(ascending=False) +
        rank_table["RFE_rank(1=best)"].rank(ascending=True)
    )
    rank_table = rank_table.sort_values("combined_score").reset_index(drop=True)
    return rank_table


In [8]:
# ----------------------------------------------------------------------
# 8. Ensemble model training + evaluation (step 5)
# ----------------------------------------------------------------------
def train_evaluate(df, feature_cols, dataset_name):
    X = df[feature_cols].values
    y = df["essential"].values

    scaler = MinMaxScaler()
    Xs = scaler.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(
        Xs, y, test_size=0.25, stratify=y, random_state=42
    )

    rf = RandomForestClassifier(n_estimators=400, class_weight="balanced", random_state=42)
    xgb = XGBClassifier(
        n_estimators=400, max_depth=4, eval_metric="logloss",
        scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
        random_state=42, verbosity=0
    )
    lr = LogisticRegression(max_iter=3000, class_weight="balanced")

    ensemble = VotingClassifier(
        estimators=[("rf", rf), ("xgb", xgb), ("lr", lr)],
        voting="soft", weights=[2, 2, 1]
    )
    ensemble.fit(X_train, y_train)

    y_pred = ensemble.predict(X_test)
    y_proba = ensemble.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred, target_names=["non-essential", "essential"])

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)

    prec, rec, _ = precision_recall_curve(y_test, y_proba)
    ap = average_precision_score(y_test, y_proba)

    print(f"\n[{dataset_name}] Ensemble evaluation")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  MSE      : {mse:.4f}")
    print(f"  ROC-AUC  : {roc_auc:.4f}")
    print(f"  PR-AUC   : {ap:.4f}")
    print("  Confusion matrix:\n", cm)
    print(report)

    return {
        "dataset": dataset_name,
        "accuracy": acc,
        "mse": mse,
        "roc_auc": roc_auc,
        "pr_auc": ap,
        "fpr": fpr, "tpr": tpr,
        "precision": prec, "recall": rec,
        "confusion_matrix": cm,
        "report": report,
        "n_test": len(y_test),
        "n_essential_test": int(y_test.sum()),
    }




In [16]:
# ----------------------------------------------------------------------
# 9. Main driver
# ----------------------------------------------------------------------
def main():
    essential_set = load_essential_set(UPLOAD_DIR)
    sl_map = load_subcellular_map(UPLOAD_DIR)
    print(f"Essential reference proteins: {len(essential_set)}")
    print(f"Proteins with subcellular annotation: {len(sl_map)}")

    all_feature_tables = {}
    all_rank_tables = {}
    all_results = {}

    base_features = ["CC", "ECC", "SL_Nb", "GO_Nb"]

    for ds in DATASETS:
        df = build_feature_table(ds, essential_set, sl_map, UPLOAD_DIR)
        all_feature_tables[ds] = df

        rank_table = rank_features(df, base_features)
        all_rank_tables[ds] = rank_table
        print(f"\n[{ds}] Feature ranking:\n{rank_table}")

        # Final dataset: ranked base features + MEM-FET engineered feature
        final_features = base_features + ["MEM_FET"]
        df.to_csv(os.path.join(OUT_DIR, f"{ds}_feature_table.csv"), index=False)
        rank_table.to_csv(os.path.join(OUT_DIR, f"{ds}_feature_ranking.csv"), index=False)

        result = train_evaluate(df, final_features, ds)
        all_results[ds] = result

    # -------------------- Summary metrics table --------------------
    summary = pd.DataFrame([{
        "Dataset": ds,
        "Nodes": len(all_feature_tables[ds]),
        "Essential(labelled)": int(all_feature_tables[ds]["essential"].sum()),
        "Accuracy": all_results[ds]["accuracy"],
        "MSE": all_results[ds]["mse"],
        "ROC_AUC": all_results[ds]["roc_auc"],
        "PR_AUC": all_results[ds]["pr_auc"],
    } for ds in DATASETS])
    summary.to_csv(os.path.join(OUT_DIR, "summary_metrics.csv"), index=False)
    print("\n=== SUMMARY ===")
    print(summary.to_string(index=False))

    # -------------------- ROC curve (all datasets) --------------------
    plt.figure(figsize=(7, 6))
    for ds in DATASETS:
        r = all_results[ds]
        plt.plot(r["fpr"], r["tpr"], label=f"{ds} (AUC={r['roc_auc']:.3f})", linewidth=2)
    plt.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve — Essential Protein Prediction (MEM-FET Ensemble)")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "ROC_curves_all_datasets.png"), dpi=150)
    plt.close()

    # -------------------- Precision-Recall curve (all datasets) --------------------
    plt.figure(figsize=(7, 6))
    for ds in DATASETS:
        r = all_results[ds]
        plt.plot(r["recall"], r["precision"], label=f"{ds} (AP={r['pr_auc']:.3f})", linewidth=2)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall Curve — Essential Protein Prediction (MEM-FET Ensemble)")
    plt.legend(loc="upper right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "PR_curves_all_datasets.png"), dpi=150)
    plt.close()

    # -------------------- Accuracy / AUC bar chart --------------------
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(DATASETS))
    width = 0.25
    ax.bar(x - width, summary["Accuracy"], width, label="Accuracy")
    ax.bar(x, summary["ROC_AUC"], width, label="ROC-AUC")
    ax.bar(x + width, summary["PR_AUC"], width, label="PR-AUC")
    ax.set_xticks(x)
    ax.set_xticklabels(DATASETS)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Score")
    ax.set_title("Model Performance Across Datasets")
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "performance_comparison.png"), dpi=150)
    plt.close()

    print(f"\nAll outputs saved to {OUT_DIR}")
    return all_feature_tables, all_rank_tables, all_results, summary


if __name__ == "__main__":
    main()

Essential reference proteins: 1285
Proteins with subcellular annotation: 4081

[YDIP] building network & features ...
  nodes=5093  edges=24743

[YDIP] Feature ranking:
  feature  RF_importance  XGB_importance  RFE_rank(1=best)  combined_score
0     ECC       0.363151        0.406465                 1             3.0
1   GO_Nb       0.236961        0.209850                 4             8.0
2   SL_Nb       0.204513        0.204324                 3             9.0
3      CC       0.195376        0.179361                 2            10.0

[YDIP] Ensemble evaluation
  Accuracy : 0.7661
  MSE      : 0.2339
  ROC-AUC  : 0.6041
  PR-AUC   : 0.4002
  Confusion matrix:
 [[892  90]
 [208  84]]
               precision    recall  f1-score   support

non-essential       0.81      0.91      0.86       982
    essential       0.48      0.29      0.36       292

     accuracy                           0.77      1274
    macro avg       0.65      0.60      0.61      1274
 weighted avg       0.74   